# 02 - Limpeza e Pré-processamento dos Dados

Neste notebook realizamos a limpeza e preparação dos dados para modelagem.

## Importar bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Carregar dados originais

In [2]:
df = pd.read_csv('../data/raw/dataset.csv')
print(f"Dataset original: {df.shape}")
print(f"\nPrimeiras linhas:")
print(df.head())

Dataset original: (9000, 27)

Primeiras linhas:
   patient_id  age     sex  resting_bp_systolic  resting_bp_diastolic  \
0           1   44    Male                  117                    74   
1           2   57    Male                  139                    94   
2           3   29    Male                  128                    78   
3           4   72    Male                  132                    86   
4           5   62  Female                  116                    75   

   cholesterol_total  hdl  ldl  triglycerides  fasting_blood_sugar  ...  \
0                193   57  106            119                  112  ...   
1                185   69  110             35                  114  ...   
2                197   52  108            157                   95  ...   
3                197   59  104            143                   92  ...   
4                154   65   75            104                  135  ...   

   family_history  smoker_status  alcohol_units_per_week  \
0 

## 1. Tratamento de Valores Ausentes

In [3]:
# Verificar valores ausentes
print("Valores ausentes no dataset:")
print(df.isnull().sum())
print(f"\nTotal de valores ausentes: {df.isnull().sum().sum()}")

Valores ausentes no dataset:
patient_id                   0
age                          0
sex                          0
resting_bp_systolic          0
resting_bp_diastolic         0
cholesterol_total            0
hdl                          0
ldl                          0
triglycerides                0
fasting_blood_sugar          0
hba1c                        0
bmi                          0
resting_heart_rate           0
max_heart_rate_achieved      0
chest_pain_type              0
exercise_induced_angina      0
st_depression                0
family_history               0
smoker_status                0
alcohol_units_per_week       0
exercise_minutes_per_week    0
sleep_hours                  0
stress_score                 0
wearable_owner               0
daily_steps                  0
diet_quality_score           0
has_heart_disease            0
dtype: int64

Total de valores ausentes: 0


In [4]:
# Como não temos valores ausentes, criamos uma cópia para trabalhar
df_clean = df.copy()

print("✓ Dataset sem valores ausentes - nenhum tratamento necessário")

✓ Dataset sem valores ausentes - nenhum tratamento necessário


## 2. Tratamento de Outliers

In [5]:
# Função para detectar outliers usando IQR
def detectar_outliers_iqr(data, coluna):
    Q1 = data[coluna].quantile(0.25)
    Q3 = data[coluna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    return (data[coluna] < limite_inf) | (data[coluna] > limite_sup)

# Detectar outliers em variáveis numéricas
numericas = df_clean.select_dtypes(include=[np.number]).columns
outliers_encontrados = {}

for col in numericas:
    outliers = detectar_outliers_iqr(df_clean, col)
    outliers_encontrados[col] = outliers.sum()

print("Outliers detectados por coluna:")
for col, count in sorted(outliers_encontrados.items(), key=lambda x: x[1], reverse=True):
    if count > 0:
        pct = (count / len(df_clean)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")

Outliers detectados por coluna:
  st_depression: 485 (5.4%)
  alcohol_units_per_week: 431 (4.8%)
  sleep_hours: 86 (1.0%)
  resting_bp_diastolic: 75 (0.8%)
  stress_score: 71 (0.8%)
  diet_quality_score: 69 (0.8%)
  resting_bp_systolic: 66 (0.7%)
  hdl: 63 (0.7%)
  cholesterol_total: 50 (0.6%)
  triglycerides: 42 (0.5%)
  resting_heart_rate: 42 (0.5%)
  bmi: 39 (0.4%)
  ldl: 34 (0.4%)
  fasting_blood_sugar: 32 (0.4%)
  daily_steps: 32 (0.4%)
  exercise_minutes_per_week: 27 (0.3%)
  max_heart_rate_achieved: 17 (0.2%)
  hba1c: 10 (0.1%)


In [6]:
# Decisão: MANTER os outliers (são dados reais e podem ser importantes)
# Justificativa: Em dados de saúde, outliers frequentemente representam casos reais
# (pacientes com condições extremas) e não devem ser removidos sem análise cuidadosa.

print("\n✓ Decisão: MANTER os outliers")
print("\nJustificativa:")
print("- Outliers em dados médicos são frequentemente casos reais e importantes")
print("- Remover seria perder informações clínicas relevantes")
print("- A modelagem deve aprender a lidar com variabilidade natural")


✓ Decisão: MANTER os outliers

Justificativa:
- Outliers em dados médicos são frequentemente casos reais e importantes
- Remover seria perder informações clínicas relevantes
- A modelagem deve aprender a lidar com variabilidade natural


## 3. Codificação de Variáveis Categóricas

In [7]:
# Identificar variáveis categóricas
categoricas = df_clean.select_dtypes(include=['object']).columns
print("Variáveis categóricas encontradas:")
print(list(categoricas))

for col in categoricas:
    print(f"\n{col}: {df_clean[col].unique()}")

Variáveis categóricas encontradas:
['sex', 'chest_pain_type', 'smoker_status']

sex: ['Male' 'Female']

chest_pain_type: ['Asymptomatic' 'Non-Anginal Pain' 'Atypical Angina' 'Typical Angina']

smoker_status: ['Never' 'Current' 'Former']


In [8]:
# Aplicar codificação
print("\nAplicando codificações:")
print("\n1. sex (Sexo) - Label Encoding")
# Sexo: Male -> 1, Female -> 0
df_clean['sex'] = (df_clean['sex'] == 'Male').astype(int)
print(f"   Male=1, Female=0")
print(f"   Resultado: {df_clean['sex'].unique()}")


Aplicando codificações:

1. sex (Sexo) - Label Encoding
   Male=1, Female=0
   Resultado: [1 0]


In [9]:
print("\n2. chest_pain_type (Tipo de Dor no Peito) - One-Hot Encoding")
# Criar variáveis dummy para tipo de dor no peito
chest_dummies = pd.get_dummies(df_clean['chest_pain_type'], prefix='chest_pain', drop_first=False)
print(f"   Categorias: {df_clean['chest_pain_type'].unique()}")
print(f"   Novas colunas: {list(chest_dummies.columns)}")

df_clean = pd.concat([df_clean, chest_dummies], axis=1)
df_clean = df_clean.drop('chest_pain_type', axis=1)


2. chest_pain_type (Tipo de Dor no Peito) - One-Hot Encoding
   Categorias: ['Asymptomatic' 'Non-Anginal Pain' 'Atypical Angina' 'Typical Angina']
   Novas colunas: ['chest_pain_Asymptomatic', 'chest_pain_Atypical Angina', 'chest_pain_Non-Anginal Pain', 'chest_pain_Typical Angina']


In [11]:
print("\n3. smoker_status (Status de Fumante) - One-Hot Encoding")
# Criar variáveis dummy para status de fumante
smoker_dummies = pd.get_dummies(df_clean['smoker_status'], prefix='smoker', drop_first=False)
print(f"   Categorias: {df_clean['smoker_status'].unique()}")
print(f"   Novas colunas: {list(smoker_dummies.columns)}")

df_clean = pd.concat([df_clean, smoker_dummies], axis=1)
df_clean = df_clean.drop('smoker_status', axis=1)


3. smoker_status (Status de Fumante) - One-Hot Encoding


In [12]:
# Variáveis booleanas (True/False) -> 1/0
print("\n4. Variáveis booleanas - Conversão para 0/1")
boolean_cols = df_clean.select_dtypes(include=['bool']).columns
for col in boolean_cols:
    df_clean[col] = df_clean[col].astype(int)
    print(f"   {col}: True->1, False->0")

print(f"\n✓ Colunas booleanas convertidas: {list(boolean_cols)}")


4. Variáveis booleanas - Conversão para 0/1
   exercise_induced_angina: True->1, False->0
   family_history: True->1, False->0
   wearable_owner: True->1, False->0
   chest_pain_Asymptomatic: True->1, False->0
   chest_pain_Atypical Angina: True->1, False->0
   chest_pain_Non-Anginal Pain: True->1, False->0
   chest_pain_Typical Angina: True->1, False->0
   smoker_Current: True->1, False->0
   smoker_Former: True->1, False->0
   smoker_Never: True->1, False->0

✓ Colunas booleanas convertidas: ['exercise_induced_angina', 'family_history', 'wearable_owner', 'chest_pain_Asymptomatic', 'chest_pain_Atypical Angina', 'chest_pain_Non-Anginal Pain', 'chest_pain_Typical Angina', 'smoker_Current', 'smoker_Former', 'smoker_Never']


In [13]:
# Verificar se ainda existem variáveis categorias
categoricas_restantes = df_clean.select_dtypes(include=['object']).columns
print(f"\nVariáveis categóricas restantes: {list(categoricas_restantes)}")


Variáveis categóricas restantes: []


## 4. Normalização/Padronização

In [14]:
# Verificar escala das variáveis numéricas
print("Estatísticas das variáveis numéricas:")
print(df_clean.describe())

Estatísticas das variáveis numéricas:
        patient_id          age          sex  resting_bp_systolic  \
count  9000.000000  9000.000000  9000.000000          9000.000000   
mean   4500.500000    53.977333     0.525667           127.849778   
std    2598.220545    13.008051     0.499369            13.767233   
min       1.000000    18.000000     0.000000            85.000000   
25%    2250.750000    45.000000     0.000000           119.000000   
50%    4500.500000    54.000000     1.000000           128.000000   
75%    6750.250000    63.000000     1.000000           137.000000   
max    9000.000000    90.000000     1.000000           181.000000   

       resting_bp_diastolic  cholesterol_total          hdl          ldl  \
count           9000.000000        9000.000000  9000.000000  9000.000000   
mean              80.788778         188.753556    55.215333   103.275556   
std               10.939943          32.271644    12.394400    27.342198   
min               50.000000         

In [15]:
# Aplicar Z-score normalization para variáveis numéricas
print("\nAplicando padronização Z-score...")

# Excluir patient_id e has_heart_disease (estas não devem ser padronizadas)
colunas_para_padronizar = [col for col in df_clean.columns 
                           if col not in ['patient_id', 'has_heart_disease']]

scaler = StandardScaler()
df_scaled = df_clean.copy()
df_scaled[colunas_para_padronizar] = scaler.fit_transform(df_clean[colunas_para_padronizar])

print(f"✓ Padronização aplicada a {len(colunas_para_padronizar)} colunas")
print(f"\nEstatísticas após padronização:")
print(df_scaled[colunas_para_padronizar].describe())


Aplicando padronização Z-score...
✓ Padronização aplicada a 30 colunas

Estatísticas após padronização:
                age           sex  resting_bp_systolic  resting_bp_diastolic  \
count  9.000000e+03  9.000000e+03         9.000000e+03          9.000000e+03   
mean  -5.921189e-17  5.447494e-17         2.629008e-16         -3.268497e-16   
std    1.000056e+00  1.000056e+00         1.000056e+00          1.000056e+00   
min   -2.765928e+00 -1.052721e+00        -3.112619e+00         -2.814502e+00   
25%   -6.901750e-01 -1.052721e+00        -6.428503e-01         -6.205841e-01   
50%    1.742607e-03  9.499191e-01         1.091218e-02          1.930851e-02   
75%    6.936602e-01  9.499191e-01         6.646746e-01          6.592011e-01   
max    2.769413e+00  9.499191e-01         3.860847e+00          4.132904e+00   

       cholesterol_total           hdl           ldl  triglycerides  \
count       9.000000e+03  9.000000e+03  9.000000e+03   9.000000e+03   
mean        1.563194e-16 -2.3684

## 5. Salvar Datasets Processados

In [16]:
# Salvar dataset limpo (sem padronização)
df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print("✓ Salvo: dataset_clean.csv")
print(f"  Dimensões: {df_clean.shape}")
print(f"  Colunas: {list(df_clean.columns)[:5]}... (total: {len(df_clean.columns)})")

✓ Salvo: dataset_clean.csv
  Dimensões: (9000, 32)
  Colunas: ['patient_id', 'age', 'sex', 'resting_bp_systolic', 'resting_bp_diastolic']... (total: 32)


In [17]:
# Salvar dataset padronizado (para modelos que se beneficiam disso)
df_scaled.to_csv('../data/processed/dataset_scaled.csv', index=False)
print("\n✓ Salvo: dataset_scaled.csv")
print(f"  Dimensões: {df_scaled.shape}")
print(f"  (Versão padronizada para modelos baseados em distância como KNN, SVM, etc.)")


✓ Salvo: dataset_scaled.csv
  Dimensões: (9000, 32)
  (Versão padronizada para modelos baseados em distância como KNN, SVM, etc.)


In [18]:
# Preparar datasets específicos para regressão
# Remover linhas onde max_heart_rate_achieved é NaN (se houver)
df_regression = df_clean.dropna(subset=['max_heart_rate_achieved'])
df_regression.to_csv('../data/processed/dataset_regression.csv', index=False)
print("\n✓ Salvo: dataset_regression.csv")
print(f"  Dimensões: {df_regression.shape}")
print(f"  Variável alvo: max_heart_rate_achieved")


✓ Salvo: dataset_regression.csv
  Dimensões: (9000, 32)
  Variável alvo: max_heart_rate_achieved


In [19]:
# Preparar datasets específicos para classificação
df_classification = df_clean.dropna(subset=['has_heart_disease'])
df_classification.to_csv('../data/processed/dataset_classification.csv', index=False)
print("\n✓ Salvo: dataset_classification.csv")
print(f"  Dimensões: {df_classification.shape}")
print(f"  Variável alvo: has_heart_disease")
print(f"  Distribuição das classes:")
print(f"    Sem doença (0): {(df_classification['has_heart_disease']==0).sum()}")
print(f"    Com doença (1): {(df_classification['has_heart_disease']==1).sum()}")


✓ Salvo: dataset_classification.csv
  Dimensões: (9000, 32)
  Variável alvo: has_heart_disease
  Distribuição das classes:
    Sem doença (0): 6273
    Com doença (1): 2727


## Resumo do Pré-processamento

**Ações realizadas:**
1. ✓ Valores ausentes: Nenhum encontrado
2. ✓ Outliers: Mantidos (dados reais em saúde)
3. ✓ Variáveis categóricas: Codificadas
   - sex: Label Encoding (Male=1, Female=0)
   - chest_pain_type: One-Hot Encoding
   - smoker_status: One-Hot Encoding
   - Booleanas: Conversão para 0/1
4. ✓ Normalização: Z-score aplicado
5. ✓ Datasets salvos em `data/processed/`
